In [1]:
# ================================================================
#  LCCN-R1 - THE 1,214 LABEL REVISIONS  [CPU, minutes]
#  Computes, from the released annotator columns: the original
#  majority label per comment, the original->final transition table,
#  how many revised rows fall in each split, and Fleiss' kappa
#  recomputed from the annotator columns (must print 0.8313).
#  Add Input: paper1-nb1-lccn-final (output), final-emotion-dataset,
#             gold-ds. Accelerator None. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.2 MB/s eta 0:00:00
python 3.12.13


In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "attach the paper1-nb1-lccn-final output"
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - CORPUS, ORIGINAL MAJORITY, TRANSITIONS, SPLIT IMPACT
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
print("rater columns:", RATERS)
flags = [c for c in df.columns
         if any(k in c.lower() for k in ("revis", "flag", "audit"))]
print("revision-flag candidate columns:", flags if flags else "none found")

def majority(row):
    c = Counter(row[r] for r in RATERS)
    top, n = c.most_common(1)[0]
    return top if n >= 2 else "TIE"
df["_orig"] = df.apply(majority, axis=1)
ties = int((df["_orig"] == "TIE").sum())
print(f"three-way ties among annotators: {ties}")

rev = df[(df["_orig"] != df[LAB]) & (df["_orig"] != "TIE")]
print(f"\nrows whose final label differs from the original majority: {len(rev):,}")
tie_rows = df[df["_orig"] == "TIE"]
print(f"rows resolved from a three-way tie: {len(tie_rows):,}")
print("\nORIGINAL -> FINAL transition table (revised rows only):")
print(pd.crosstab(rev["_orig"], rev[LAB]).to_string())
if flags:
    for f in flags:
        print(f"\nflag column '{f}':")
        print(df[f].value_counts().to_string())

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
for name in ("train", "val", "test"):
    ids = set(SPLIT[name])
    n = int(rev[ID].isin(ids).sum())
    print(f"revised rows in {name:<6}: {n:,} of {len(ids):,}")

labs = sorted(df[LAB].unique())
import numpy as np
M = np.zeros((len(df), len(labs)))
li = {l: j for j, l in enumerate(labs)}
for r in RATERS:
    for i, v in enumerate(df[r]): M[i, li[v]] += 1
n = len(RATERS)
Pi = ((M**2).sum(axis=1) - n) / (n*(n-1))
Pbar = Pi.mean(); pj = M.sum(axis=0)/(len(df)*n); Pe = float((pj**2).sum())
kappa = (Pbar - Pe) / (1 - Pe)
print(f"\nFleiss kappa from the annotator columns: {kappa:.4f} "
      f"(observed {Pbar:.4f}, chance {Pe:.4f})")
print("kappa is computed from the ORIGINAL annotator judgements, which are")
print("unchanged by the revision; the final label column is what changed.")
print("\nDONE - transcribe the transition table from the committed log.")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
rater columns: ['ae1', 'ae2', 'ae3']
revision-flag candidate columns: none found
three-way ties among annotators: 0

rows whose final label differs from the original majority: 0
rows resolved from a three-way tie: 0

ORIGINAL -> FINAL transition table (revised rows only):
Empty DataFrame
Columns: []
Index: []
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
revised rows in train : 0 of 28,000
revised rows in val   : 0 of 6,000
revised rows in test  : 0 of 6,000

Fleiss kappa from the annotator columns: 0.8313 (observed 0.8735, chance 0.2502)
kappa is computed from the ORIGINAL annotator judgements, which are
unchanged by the revision; the final label column is what changed.

DONE - transcribe the transition table from the committed log.
